In [1]:
from model_v2_w6ms3.My.Encoder_sober import *
from model_v2_w6ms3.My.train_v22_hard_fbmask import *

from model_v2_w6ms3.DataLoader import TempFlowDataset_disp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
cfg = {
  "data": {
    "data_root": "../../Data",
    "stats_file": "stats.json",
    "crop_size": [
      352,
      1216
    ],
    "seq_len": 4,
    "center_frame_idx": 10,
    "split": "training",
    "image_folder": "image_2",
    "flow_type": "flow_occ",
    "disp_type": "disp_occ",
    "normalize": True,
    "return_pair_only": False
  },
  "model": {
    "pair_feat_ch": 64,
    "corr_radius": 4,
    "pair_embed_ch": 128,
    "predict_flow_init": True,
    "visual_in_ch": 3,
    "visual_base_ch": 32,
    "visual_out_ch": 64,
    "motion_in_ch": 128,
    "motion_hidden_ch": 128,
    "motion_out_ch": 64,
    "fusion_visual_ch": 64,
    "fusion_motion_ch": 64,
    "fusion_hidden_ch": 128,
    "fusion_out_ch": 128,
    "uno_use_valid_mask": True,
    "uno_hidden_channels": 64,
    "uno_lifting_channels": 128,
    "uno_projection_channels": 128,
    "uno_n_layers": 4,
    "uno_out_channels": [
      64,
      128,
      128,
      64
    ],
    "uno_n_modes": [
      [
        16,
        16
      ],
      [
        12,
        12
      ],
      [
        12,
        12
      ],
      [
        16,
        16
      ]
    ],
    "uno_scalings": [
      [
        1.0,
        1.0
      ],
      [
        0.5,
        0.5
      ],
      [
        2.0,
        2.0
      ],
      [
        1.0,
        1.0
      ]
    ],
    "decoder_in_ch": 128,
    "decoder_hidden_ch": 64,
    "decoder_upsample": 8,
    "decoder_use_prev_flow": True
  },
  "train": {
    "batch_size": 2,
    "num_epochs": 400,
    "lr": 0.0001,
    "weight_decay": 0.0001,
    "num_workers": 0,
    "shuffle": True,
    "pin_memory": True,
    "log_every_n_steps": 10,
    "seed": 42,
    "save_epoch_checkpoints": [
      50,
      100,
      150,
      200
    ]
  },
  "loss": {
    "lambda_temp": 0.03,
    "lambda_smooth": 0.01,
    "lambda_self": 0.1,
    "lambda_edge_weight": 1.0,
    "photo_lambda_l1": 0.15,
    "photo_lambda_ssim": 0.85,
    "photo_lambda_census": 0.3,
    "photo_census_patch": 7,
    "photo_multiframe_reduction": "mean",
    "photo_use_confidence": True,
    "photo_texture_floor": 0.05,
    "photo_robust_beta": 10.0,
    "photo_use_fb_consistency": True,
    "photo_bidirectional": True,
    "fb_alpha": 0.01,
    "fb_beta": 0.5,
    "fb_gamma": 2.0,
    "fb_conf_floor": 0.05,
    "lambda_acc_photo": 0.05,
    "lambda_acc_smooth": 0.0,
    "accum_max_skip": 2,
    "lambda_fb": 0.01,
    "fb_loss_robust_eps": 0.01,
    "photo_use_hard_fb_mask": True,
    "fb_mask_alpha1": 0.01,
    "fb_mask_alpha2": 0.5
  },
  "experiment": {
    "experiment_name": "v22_hard_fbmask",
    "save_dir": "checkpoints",
    "tensorboard_dir": "runs",
    "checkpoint_name": "fullpipeline_v22_latest.pth",
    "best_checkpoint_name": "fullpipeline_v22_best.pth",
    "config_dump_name": "config_v22_hard_fbmask.json"
  }
}

In [3]:
set_seed(cfg["train"]["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = TempFlowDataset_disp(
        root=cfg["data"]["data_root"],
        split=cfg["data"]["split"],
        image_folder=cfg["data"]["image_folder"],
        flow_type=cfg["data"]["flow_type"],
        disp_type=cfg["data"]["disp_type"],
        seq_len=cfg["data"]["seq_len"],
        center_frame_idx=cfg["data"]["center_frame_idx"],
        crop_size=tuple(cfg["data"]["crop_size"]),
        normalize=cfg["data"]["normalize"],
        stats_in=cfg["data"]["stats_file"],
        return_pair_only=cfg["data"]["return_pair_only"],
    )

train_loader = DataLoader(
        dataset,
        batch_size=cfg["train"]["batch_size"],
        shuffle=cfg["train"]["shuffle"],
        num_workers=cfg["train"]["num_workers"],
        pin_memory=cfg["train"]["pin_memory"],
    )

modules = build_modules(cfg, device)
optimizer = torch.optim.AdamW(
        [p for module in modules.values() for p in module.parameters()],
        lr=1e-4,
        weight_decay=1e-4
    )


In [4]:
def forward_pipeline(modules, imgs, valid, uno_use_valid_mask=True):
        pair_encoder = modules['pair_encoder'].to(device)
        pair_out = pair_encoder(imgs)
        #pair_out = modules["pair_encoder"](imgs).to(device)
        pair_feats = pair_out["pair_feats"].to(device)
        flow_inits = pair_out["flow_inits"].to(device)
        corrs = pair_out["corrs"].to(device)

        if flow_inits is None:
            raise RuntimeError("v19 UNO integration requires predict_flow_init=True in the pair encoder.")

        visual_feats = modules["visual_branch"](imgs)
        motion_feats = modules["motion_branch"](pair_feats)
        fused_seq = modules["fusion"](visual_feats, motion_feats)

        valid_ds = None
        if uno_use_valid_mask:
            valid_ds = downsample_valid_mask(valid, fused_seq.shape[-2:])

        uno_in = build_uno_input_2d(fused_seq, flow_inits, valid_mask=valid_ds)
        uno_feat = modules["uno"](uno_in)

        b, tm, _, h, w = fused_seq.shape
        latent_delta = modules["latent_head"](uno_feat, b, tm, h, w)
        refined_seq = fused_seq + latent_delta

        flows, flow_residuals = modules["decoder"](refined_seq, flow_inits=flow_inits)

        return {
            "flows": flows,
            "flow_inits": flow_inits,
            "pair_feats": pair_feats,
            "corrs": corrs,
            "fused_seq": fused_seq,
            "latent_delta": latent_delta,
            "refined_seq": refined_seq,
            "flow_residuals": flow_residuals,
        }

In [5]:
def load_checkpoint(ckpt_path, modules, device, optimizer=None, strict=True):
    ckpt = torch.load(ckpt_path, map_location=device)

    # ---- load module weights ----
    for name, module in modules.items():
        key = f"{name}_state_dict"
        if key in ckpt:
            module.load_state_dict(ckpt[key], strict=strict)
        else:
            print(f"[Warning] Missing key: {key}")

        module.to(device)

    # ---- load optimizer (optional) ----
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    # ---- metadata ----
    epoch = ckpt.get("epoch", None)
    stats = ckpt.get("stats", None)
    config = ckpt.get("config", None)

    print(f"Loaded checkpoint complete from: {ckpt_path} (epoch={epoch})")

    return {
        "epoch": epoch,
        "stats": stats,
        "config": config,
    }

In [6]:
meta = load_checkpoint(
    ckpt_path="../../model_v2_w6ms3/My/fullpipeline_v22_best.pth",
    modules=modules,
    device=device,
    optimizer=optimizer,   # optional
)
for m in modules.values():
    m.to(device)

Loaded checkpoint complete from: ../../model_v2_w6ms3/My/fullpipeline_v22_best.pth (epoch=387)


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import torch


def flow_to_rgb(flow, max_mag=None):
    if isinstance(flow, torch.Tensor):
        flow = flow.detach().cpu().numpy()

    u = flow[0]
    v = flow[1]

    mag = np.sqrt(u**2 + v**2)
    ang = np.arctan2(v, u)

    hue = (ang + np.pi) / (2 * np.pi)
    sat = np.ones_like(hue)

    if max_mag is None:
        max_mag = np.max(mag) + 1e-6

    val = np.clip(mag / max_mag, 0, 1)

    hsv = np.stack([hue, sat, val], axis=-1)

    import matplotlib.colors as mcolors
    rgb = mcolors.hsv_to_rgb(hsv)
    return rgb


def select_gt_flow_single(pred_flows, src_idx, b=0):
    """
    pred_flows: [B, Tm, 2, H, W]
    src_idx: [B]
    b: sample index in batch
    """
    t = int(src_idx[b].item())
    return pred_flows[b, t]

In [8]:
import os
import matplotlib.pyplot as plt

def save_all_flow_preds(batch, pred_flows, idx, save_root="Tests/MS3/preds"):
    """
    pred_flows: [B, T, 2, H, W]

    Saves as:
    Tests/MS3/preds/seq_{idx}_{b}/00000.png
    Tests/MS3/preds/seq_{idx}_{b}/00001.png
    Tests/MS3/preds/seq_{idx}_{b}/00002.png
    """
    B, T = pred_flows.shape[:2]

    for b in range(B):
        seq_dir = os.path.join(save_root, f"seq_{idx}_{b}")
        os.makedirs(seq_dir, exist_ok=True)

        for t in range(T):
            pred = pred_flows[b, t].detach().cpu()
            pred_rgb = flow_to_rgb(pred)

            pred_rgb = pred_rgb.astype(float)
            if pred_rgb.max() > 1.0:
                pred_rgb = pred_rgb / 255.0

            fig, ax = plt.subplots()
            ax.imshow(pred_rgb)
            ax.axis("off")
            plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

            save_path = os.path.join(seq_dir, f"{t:05d}.png")
            fig.savefig(save_path, bbox_inches="tight", pad_inches=0)
            plt.close(fig)

In [9]:
batch = next(iter(train_loader))
out = forward_pipeline(modules=modules, imgs=batch["imgs"].to(device), valid=batch["valid"].to(device))
pred_flows = out['flows']
#visualize_batch_result_with_save(batch, pred_flows, sample_idx=0)

In [10]:
print(pred_flows.shape)

torch.Size([2, 3, 2, 352, 1216])


In [11]:
for i in range(15):
    batch = next(iter(train_loader))

    with torch.no_grad():
        out = forward_pipeline(
            modules=modules,
            imgs=batch["imgs"].to(device),
            valid=batch["valid"].to(device)
        )

    pred_flows = out["flows"]
    save_all_flow_preds(batch, pred_flows, idx=i, save_root="Tests/MS3/preds")

    print(f"#{i} done")

#0 done
#1 done
#2 done
#3 done
#4 done
#5 done
#6 done
#7 done
#8 done
#9 done
#10 done
#11 done
#12 done
#13 done
#14 done
